# Verificación del agente Paper

**Juan Fernando Castrejón Villa · 81901**

Este cuaderno comprueba los contratos de los 15 modelos y lee las pruebas y la comparación histórica incluidas en el proyecto. Abrir desde la carpeta extraída con el entorno de Python del README.

In [1]:
from pathlib import Path
import os,sys,json,subprocess
RUTA=Path(os.environ.get('PAI_PROJECT_ROOT',str(Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()))).resolve()
if not (RUTA/'pai/config.py').exists():raise RuntimeError('Abrir este notebook dentro del proyecto extraído o definir PAI_PROJECT_ROOT.')
sys.path.insert(0,str(RUTA));os.chdir(RUTA)
import pandas as pd
from IPython.display import display,Markdown
from pai.config import symbols,PaiError
from backend.models import Registry
registry=Registry(RUTA);print('Proyecto:', RUTA.name)

Proyecto: PAI_81901


In [2]:
estado=[]
for symbol in symbols():
    try:
        m=registry.contract(symbol);estado.append({'symbol':symbol,'modelo':'Listo','version':m['version'],'motivo':'T4, hashes y TE comprobados'})
    except PaiError as exc:estado.append({'symbol':symbol,'modelo':'Pendiente','version':None,'motivo':str(exc)})
display(pd.DataFrame(estado))

,symbol,modelo,version,motivo
0,BTCUSDT,Listo,1.0.0,"T4, hashes y TE comprobados"
1,ETHUSDT,Listo,1.0.0,"T4, hashes y TE comprobados"
2,SOLUSDT,Listo,1.0.0,"T4, hashes y TE comprobados"
3,XRPUSDT,Listo,1.0.0,"T4, hashes y TE comprobados"
4,ADAUSDT,Listo,1.0.0,"T4, hashes y TE comprobados"
5,DOGEUSDT,Listo,1.0.0,"T4, hashes y TE comprobados"
6,AVAXUSDT,Listo,1.0.0,"T4, hashes y TE comprobados"
7,DOTUSDT,Listo,1.0.0,"T4, hashes y TE comprobados"
8,LINKUSDT,Listo,1.0.0,"T4, hashes y TE comprobados"
9,LTCUSDT,Listo,1.0.0,"T4, hashes y TE comprobados"


In [3]:
for nombre in ['pruebas.json', 'ollama_local.json', 'incidente_mexc.json', 'replay_resumen.json']:
    ruta = RUTA / 'evidencia' / nombre
    if not ruta.is_file():
        print(f'Archivo de evidencia ausente ({nombre}).')
        continue
    datos = json.loads(ruta.read_text(encoding='utf-8'))
    display(Markdown(f'### {nombre}'))
    if nombre == 'ollama_local.json':
        campos = ['model', 'ollama_version', 'local', 'latency_ms']
    elif nombre == 'replay_resumen.json':
        campos = ['symbol', 'mode', 'cycles', 'first_issue', 'last_target']
    else:
        campos = [campo for campo in datos if campo != 'cases']
    display(pd.DataFrame([{campo: datos.get(campo) for campo in campos}]))

### pruebas.json

,fecha,testing_only,tests,failures,errors,skipped,seconds,scope
0,2026-11-22T19:07:40.128503+00:00,True,33,0,0,0,69.025,Pruebas controladas; incluyen replay sintético...


### ollama_local.json

,model,ollama_version,local,latency_ms
0,qwen3:0.6b,0.40.16,True,10737.103743


### incidente_mexc.json

,timestamp,etapa,fuente,http_status,resultado,accion,impacto
0,2026-11-20T06:46:10.041566+00:00,conexion_MEXC,https://api.mexc.com/api/v3/time,403,Site Unavailable,No generar datos de reemplazo; volver a intent...,No se acreditan descargas de 1500 velas por ac...


### replay_resumen.json

,symbol,mode,cycles,first_issue,last_target
0,BTCUSDT,replay,30,2026-10-21 00:00:00+00:00,2026-11-20 00:00:00+00:00


## Pruebas de datos, riesgo y persistencia

El reporte de esta entrega utiliza las mismas 33 pruebas automatizadas que se ejecutaron después de incorporar los modelos.

In [4]:
reporte = json.loads((RUTA / 'evidencia' / 'verificacion_entrega.json').read_text(encoding='utf-8'))
resultado = reporte['pruebas']
if resultado['failures'] or resultado['errors']:
    raise RuntimeError('Revisar el reporte de pruebas.')
display(pd.DataFrame([resultado]))

,tests,failures,errors,skipped
0,33,0,0,0


## Replay histórico de BTCUSDT

Las tres estrategias utilizan el mismo periodo, saldo inicial, costos y límite de exposición. Cada orden simulada se conserva en la base SQLite de su estrategia.

In [5]:
carpeta_replay = RUTA / 'evidencia' / 'replay_BTCUSDT'
registro = json.loads((carpeta_replay / 'ollama_ciclos.json').read_text(encoding='utf-8'))
assert len(registro) == 30
display(pd.read_csv(carpeta_replay / 'comparacion.csv'))
display(pd.DataFrame([
    {'cierre_ms': ciclo['close_ms'], **ciclo['evidence']['response']}
    for ciclo in registro[:3]
]))

,rendimiento_neto,max_drawdown,operaciones_cerradas,win_rate,costos,latencia_llm_ms_media,acciones_rechazadas,ciclos,patrimonio_final,estrategia
0,-0.025382,0.026613,1,0.0,0.0,0.000000,0,30,48730.912271,modelo_sin_llm
1,-0.025382,0.026613,1,0.0,0.0,11397.767484,29,30,48730.912271,agente_con_llm
2,-0.025382,0.026613,1,0.0,0.0,0.000000,0,30,48730.912271,comprar_y_mantener


,cierre_ms,action,symbol,candle_close_ms,reason
0,1792540800000,BUY,BTCUSDT,1792540800000,Pronóstico 0.106% y umbral 0.100%; distancia a...
1,1792627200000,BUY,BTCUSDT,1792627200000,Pronóstico 0.107% y umbral 0.100%; distancia a...
2,1792713600000,BUY,BTCUSDT,1792713600000,Pronóstico 0.111% y umbral 0.100%; distancia a...


## Interpretación

Las tres estrategias obtuvieron el mismo rendimiento neto. Ollama propuso comprar en los 30 ciclos. El motor ejecutó la primera compra y rechazó las otras 29 porque ya había una posición. Las tres carteras liquidaron la posición al final del periodo histórico.

Consultar el informe y las decisiones registradas en evidencia/replay_BTCUSDT/.